Using GloVe 100d for balance of performance, lightweight, and suitability for binary sentiment classification

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    GRU,
    LSTM,
    Dense,
    Dropout,
    Bidirectional
)

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

Load Dataset

In [2]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

file_path = os.path.join(BASE_DIR, "data", "processed", "cleaned.csv")

df = pd.read_csv(file_path)

Data Prep

In [3]:
X = df["clean_text"].fillna("").astype(str)
y = df["sentiment"].astype(int)

print(y.value_counts())

sentiment
0    29000
1    28994
Name: count, dtype: int64


Train/Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(46395,)
(11599,)


Tokenization

In [ ]:
max_words = 20000
max_len = 300
# max_len = 150

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
# tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

print(X_train_pad.shape)

(46395, 300)


In [8]:
word_index = tokenizer.word_index
# vocab_size = len(word_index) + 1
vocab_size = max_words

print("Vocabulary size:", vocab_size)

Vocabulary size: 20000


Load Embeddings

In [9]:
embedding_index = {}

glove_path = os.path.join(
    BASE_DIR,
    "embeddings",
    "glove.6B.100d.txt"
)

with open(glove_path, encoding="utf8") as f:
    for line in f:
        values = line.split()

        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')

        embedding_index[word] = vector

print("Loaded word vectors:", len(embedding_index))

Loaded word vectors: 400000


Embedding Matrix <br>
Maps word → tokenizer ID → GloVe vector <br>
i.e. "good" → 52 → [0.1, 0.3, ...]

In [10]:
embedding_dim = 100
max_words = 20000

embedding_matrix = np.zeros((max_words, embedding_dim))

for word, i in word_index.items():

    if i >= max_words:
        continue

    embedding_vector = embedding_index.get(word)

    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

print("Embedding matrix shape:",
      embedding_matrix.shape)

print("Nonzero values:",
      np.count_nonzero(embedding_matrix))

# debugging check
print(embedding_matrix[100][:10])
print(np.sum(embedding_matrix[100]))

Embedding matrix shape: (20000, 100)
Nonzero values: 1434100
[ 0.41711    -0.10176     0.058147   -0.18332    -0.44457999 -0.17851999
 -0.34391999 -0.077147    0.58521003 -0.52752   ]
6.792773924069479


Model (HBGRU-GlobalMaxPooling)<br>
Where return_sequences=True since LSTM requires full sequence output

In [11]:
# model_hbgru = Sequential([
    
#     Embedding(
#         input_dim=max_words,
#         output_dim=embedding_dim,
#         weights=[embedding_matrix],
#         input_length=max_len,
#         trainable=True
#     ),

#     Bidirectional(GRU(128, return_sequences=True)),

#     LSTM(64),

#     Dropout(0.5),

#     Dense(64, activation='relu'),

#     Dense(1, activation='sigmoid')
# ])

# model_hbgru = Sequential([

#     Embedding(
#         input_dim=max_words,
#         output_dim=embedding_dim,
#         weights=[embedding_matrix],
#         trainable=True
#     ),

#     Bidirectional(GRU(64, return_sequences=True)),

#     LSTM(32),

#     Dropout(0.5),

#     Dense(32, activation='relu'),

#     Dense(1, activation='sigmoid')
# ])

from tensorflow.keras.layers import GlobalMaxPooling1D
model_hbgru = Sequential([

    Embedding(
        input_dim=max_words,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=False
    ),

    Bidirectional(
        GRU(64, return_sequences=True)
    ),

    GlobalMaxPooling1D(),

    Dropout(0.3),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

model_hbgru.summary()

c:\Users\USER\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,000,000 (7.63 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,000,000 (7.63 MB)

Compile

In [12]:
print(df["sentiment"].value_counts(normalize=True))

sentiment
0.0    0.500052
1.0    0.499948
Name: proportion, dtype: float64


In [13]:
# from tensorflow.keras.optimizers import Adam
# model_hbgru.compile(
#     loss='binary_crossentropy',
#     optimizer=Adam(learning_rate=0.0005),
#     metrics=['accuracy']
# )

model_hbgru.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.0001),
    metrics=['accuracy']
)

Early Stopping

In [14]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

Training

In [15]:
history_hbgru = model_hbgru.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 176s 282ms/step - accuracy: 0.6239 - loss: 0.6474 - val_accuracy: 0.7364 - val_loss: 0.5527
Epoch 2/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 165s 284ms/step - accuracy: 0.7462 - loss: 0.5155 - val_accuracy: 0.7828 - val_loss: 0.4642
Epoch 3/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 165s 284ms/step - accuracy: 0.7778 - loss: 0.4658 - val_accuracy: 0.7989 - val_loss: 0.4350
Epoch 4/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 163s 282ms/step - accuracy: 0.7904 - loss: 0.4418 - val_accuracy: 0.8132 - val_loss: 0.4166
Epoch 5/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 163s 281ms/step - accuracy: 0.8048 - loss: 0.4216 - val_accuracy: 0.8158 - val_loss: 0.4041
Epoch 6/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 172s 296ms/step - accuracy: 0.8118 - loss: 0.4093 - val_accuracy: 0.8240 - val_loss: 0.3925
Epoch 7/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 205s 353ms/step - accuracy: 0.8209 - loss: 0.3963 - val_accuracy: 0.8272 - val_loss: 0.3830
Epoch 8/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 194s 334ms/step - accuracy: 0.8276 -

Evaluation

In [16]:
y_pred = (model_hbgru.predict(X_test_pad) > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

363/363 ━━━━━━━━━━━━━━━━━━━━ 24s 63ms/step
Accuracy: 0.8380894904733167

Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.83      0.84      5800
           1       0.83      0.85      0.84      5799

    accuracy                           0.84     11599
   macro avg       0.84      0.84      0.84     11599
weighted avg       0.84      0.84      0.84     11599



In [28]:
print(X_train_pad.shape)
print(y_train.shape)

print(type(y_train.iloc[0]))
print(y_train.unique())

(46395, 300)
(46395,)
<class 'numpy.float64'>
[0. 1.]


In [29]:
print(X_train_pad[0][:20])
print(y_train.iloc[0])

[  100    52   780  1535     7   520    66     2    45 10760    54     2
   326    43     2  8092     8    22   155     3]
0.0


In [30]:
print(df["sentiment"].value_counts())

sentiment
0.0    29000
1.0    28994
Name: count, dtype: int64


In [31]:
print(len(tokenizer.word_index))

33580


In [49]:
matched = 0

for word in word_index:
    if word in embedding_index:
        matched += 1

print("Matched words:", matched)
print("Total vocab:", len(word_index))
print("Coverage:", matched / len(word_index))

Matched words: 18886
Total vocab: 33580
Coverage: 0.5624181060154854
